# Data Preprocessing

Generated from `DELE_CA1_B.ipynb` by `scripts/split_notebook.py`.


---
# 4. Data Pre-Processing

In this section, we will augment, and process our Data so as to prepare for Model Training. Throughout this section, we will also prepare our Data in 2 different methods as there are both Classification and Regression that needs to be done. Some of these operations that will be conducted are but not limited to, Data Augmentation, Tokenisation and Padding.

---
## 4.1 Data Splitting For Classification & Regression

In this sub-section, we will be spliting our Data into Two Separate DataFrame as the Data Pre-processing for Classification may be different from the Data Pre-processing for Regression. Therefore, to accomodate for both Classification & Regression to be done and subsequently evaluated, we will need to conduct the Data Pre-processing separately.

In [ ]:
# ==================== Creating Data Label ==================== #
df['source'] = 'original'

# ==================== Splitting into Two DataFrames ==================== #
df_classification = df.copy()
df_regression = df.copy()

# ==================== Regression: Drop Classification Labels ==================== #
df_regression = df_regression.drop(columns=['Sentiment_Label'], errors='ignore')

# ==================== Displaying DataFrames ==================== #
print("Classification DataFrame:")
display(df_classification.head(5).style.background_gradient(cmap="Blues"))
print("\n"*5)
print("Regression DataFrame:")
display(df_regression.head(5).style.background_gradient(cmap="Blues"))

Classification DataFrame:


,Review,Language,word_count,Sentiment_Binary,Sentiment_Label,Score,source
0,A big surprise in the middle of the film! Thrilling action!,English,11,0,Positive,0.880000,original
1,A big surprise in the plot! Thrilling action throughout.,English,9,0,Positive,0.900000,original
2,A cinematic experience that is unforgettable. I'm impressed!,English,8,0,Positive,0.900000,original
3,"A cinematic marvel! The visuals are breathtaking, and the narrative is spellbinding.",English,12,0,Positive,0.920000,original
4,A complex yet engaging plot. A major surprise awaits at the end!,English,12,0,Positive,0.940000,original








Regression DataFrame:


,Review,Language,word_count,Sentiment_Binary,Score,source
0,A big surprise in the middle of the film! Thrilling action!,English,11,0,0.880000,original
1,A big surprise in the plot! Thrilling action throughout.,English,9,0,0.900000,original
2,A cinematic experience that is unforgettable. I'm impressed!,English,8,0,0.900000,original
3,"A cinematic marvel! The visuals are breathtaking, and the narrative is spellbinding.",English,12,0,0.920000,original
4,A complex yet engaging plot. A major surprise awaits at the end!,English,12,0,0.940000,original


With reference to the output of the Two Separate DataFrames above, we are able to verify that the Data have been successfully split into 2 different parts that can be pre-processed separately.

---
## 4.2 Train-Validation-Test Split

In this sub-section, we will split our data into 3 different parts, Training Data, Validation Data and Testing Data. These data will be used to Train and subsequently, Evaluate our RNN Models. This operation will be conducted in the Code Cell below.

In [ ]:
# ==================== Filter Original (No Augmentation) ==================== #
df_class_original = df_classification[df_classification['source'] == 'original'].reset_index(drop=True)

# ==================== Initial Stratified Train/Test Split ==================== #
df_class_train_val, df_class_test = train_test_split(
    df_class_original,
    test_size=0.10,
    stratify=df_class_original['Sentiment_Binary'],
    random_state=42
)

# ==================== Stratified Split: Train/Val from Train_Val ==================== #
df_class_train, df_class_val = train_test_split(
    df_class_train_val,
    test_size=0.1111,  # 0.1111 of 90% ≈ 10% of total
    stratify=df_class_train_val['Sentiment_Binary'],
    random_state=42
)

# ==================== Filter Original (No Augmentation) ==================== #
df_reg_original = df_regression[df_regression['source'] == 'original'].reset_index(drop=True)

# ==================== Initial Train/Test Split ==================== #
df_reg_train_val, df_reg_test = train_test_split(
    df_reg_original,
    test_size=0.10,
    shuffle=True,
    random_state=42
)

# ==================== Random Split: Train/Val from Train_Val ==================== #
df_reg_train, df_reg_val = train_test_split(
    df_reg_train_val,
    test_size=0.1111,
    shuffle=True,
    random_state=42
)

# ==================== Displaying DataFrames ==================== #
print("Classification DataFrame:")
display(df_class_train.head(5).style.background_gradient(cmap="Blues"))
print("\n"*5)
print("Regression DataFrame:")
display(df_reg_train.head(5).style.background_gradient(cmap="Blues"))

Classification DataFrame:


,Review,Language,word_count,Sentiment_Binary,Sentiment_Label,Score,source
109,Kurangnya pengembangan watak membuat filem ini lemah. Tidak memuaskan.,Malay,9,1,Negative,0.250000,original
9,Adegan aksi yang dramatik dan berkesan. Terlalu banyak efek khas.,Malay,10,1,Negative,0.400000,original
92,Interesting and surprising plot. Love the twist in the story.,English,10,0,Positive,0.850000,original
30,"Avengers: Endgame is an absolute masterpiece! The film seamlessly weaves together a complex yet engaging narrative, leaving audiences on the edge of their seats. The performances from the cast are stellar, bringing emotional depth to the characters. The visual effects are nothing short of breathtaking, creating a cinematic experience that will be remembered for years to come.",English,57,0,Positive,0.950000,original
121,Pelakon memberikan persembahan yang sangat baik. Saya suka dengan penutupan ceritanya.,Malay,11,0,Positive,0.880000,original








Regression DataFrame:


,Review,Language,word_count,Sentiment_Binary,Score,source
33,Avengers: Endgame mengecewakan dan terlalu klise. Kurang memuaskan.,Malay,8,1,0.150000,original
141,Saya kira filem ini tidak sehebat yang diperkatakan. Rasa kurang puas hati.,Malay,12,1,0.300000,original
8,Adegan akhir yang sangat epik! Saya teruja dengan penutupan trilogi ini.,Malay,11,0,0.920000,original
221,The visual effects and animation in this film are truly amazing. Impressed!,English,12,0,0.900000,original
32,Avengers: Endgame mengecewakan dan membosankan. Terlalu banyak klise.,Malay,8,1,0.150000,original


With reference to the Code Cell above, we are able to verify that the operation of spliting the data have been successfully conducted and we are able to conclude the Data Pre-Processing.

---
## 4.3 Data Augmentation

In this sub-section, we will be performing Data Augmentation which will firstly balance out the Data and subseqeuntly increase the Data Volume. This will create more Training Data for the RNN Model which in turn will allow for better generalisation and higher accuracy for Output. The method used will be similar for both Classification and Regression. However, the Basis of Application will be different in regards to the Balancing of Data.

---
### 4.3.1 Classification Balancing Data Augmentation

For the Classification Dataset, we will be balancing the Class Inbalance by increasing the Data Quantity for 'Negative' and 'Neutral' Sentiments. We will be achieving the balance through the usage of T5 Paraphraser and Deep-Translator. The T5 Paraphraser will be used as a paraphrasing model to augment underrepresented sentiment categories. It operates as a pretrained transformer, applying text-to-text transformation on input strings without altering labels or requiring external inference. The Deep-Translator will be used to translate the Reviews that are in Malay to English and subsequently, back to Malay. The Deep Translator is used as the T5 Paraphraser only supports English. The Balancing Data Augmentation will be conducted in the Code Cell below.

In [ ]:
# ==================== Load T5 Paraphrasing Model ==================== #
paraphraser = pipeline("text2text-generation", model="Vamsi/T5_Paraphrase_Paws", max_length=100)

# ==================== Language-Aware Paraphrasing ==================== #
def stacked_paraphrase(text, lang):
    try:
        if lang.lower() == 'english':
            return paraphraser(f"paraphrase: {text} </s>", num_return_sequences=1, do_sample=True)[0]['generated_text']
        elif lang.lower() == 'malay':
            to_en = GoogleTranslator(source='auto', target='en').translate(text)
            paraphrased = paraphraser(f"paraphrase: {to_en} </s>", num_return_sequences=1, do_sample=True)[0]['generated_text']
            return GoogleTranslator(source='auto', target='ms').translate(paraphrased)
        else:
            return text
    except:
        return text

# ==================== Get Class Counts (Train Only) ==================== #
class_counts = Counter(df_class_train['Sentiment_Label'])
max_count = max(class_counts.values())

# ==================== Augment All Underrepresented Classes ==================== #
augmented_rows = []

for label, current_count in class_counts.items():
    samples_needed = max_count - current_count

    if current_count == 0 or samples_needed <= 0:
        continue

    df_subset = df_class_train[df_class_train['Sentiment_Label'] == label]
    rows = df_subset.sample(n=samples_needed, replace=True).to_dict(orient='records')

    for row in rows:
        new_text = stacked_paraphrase(row['Review'], row['Language'])
        new_row = row.copy()
        new_row['Review'] = new_text
        new_row['source'] = 'augmented'
        augmented_rows.append(new_row)

# ==================== Combine Augmented Data with Training Set ==================== #
df_augmented = pd.DataFrame(augmented_rows)
df_class_train_augmented = pd.concat([df_class_train, df_augmented], ignore_index=True).sample(frac=1).reset_index(drop=True)

# ==================== Plot Review Distribution (Train Only) ==================== #
fig_lang = px.histogram(
    df_class_train_augmented,
    x='Sentiment_Label',
    color='Sentiment_Label',
    title='Train Set Review Distribution (With Augmentation)',
    color_discrete_sequence=px.colors.qualitative.Bold
)

fig_lang.update_layout(
    xaxis_title='Sentiment_Label',
    yaxis_title='Number of Reviews',
    template='plotly_white',
    title_font=dict(size=20),
    showlegend=False
)

fig_lang.show()

Device set to use cuda:0


With reference to the output above, we are able to see that the Data have been successfully balanced and we can now proceed to the next Data Augmentation on the Classification Dataset to multiply it in quantity to provide larger amounts of Data for Model Training.

---
### 4.3.2 Classification Multiplication Data Augmentation

Now that the Data for each of the Sentiment Class have been balanced, we will now increase the Data Volume for all the Sentiment Classes so as to increase the Learning Capabilities of the Model. This will be achieved through using the T5 Paraphraser and Deep-Translator but this time for a different purpose which is to Multiply instead of Balancing. We will be multiplying the Data by 5 Times.

In [ ]:
# ==================== Load T5 Paraphrasing Model ==================== #
paraphraser = pipeline("text2text-generation", model="Vamsi/T5_Paraphrase_Paws", max_length=100)

# ==================== Language-Aware Multiple Paraphrasing ==================== #
def generate_multiple_paraphrases(row, n=3):
    text = row['Review']
    lang = row['Language']
    paraphrases = []

    try:
        if lang.lower() == 'english':
            results = paraphraser(
                f"paraphrase: {text} </s>",
                num_return_sequences=n,
                do_sample=True,
                top_k=50,
                top_p=0.95
            )
            paraphrases = [res['generated_text'] for res in results]

        elif lang.lower() == 'malay':
            to_en = GoogleTranslator(source='auto', target='en').translate(text)
            results = paraphraser(
                f"paraphrase: {to_en} </s>",
                num_return_sequences=n,
                do_sample=True,
                top_k=50,
                top_p=0.95
            )
            paraphrases = [GoogleTranslator(source='auto', target='ms').translate(res['generated_text']) for res in results]
        else:
            paraphrases = [text] * n

    except:
        paraphrases = [text] * n

    return paraphrases

# ==================== Apply Multiplication to Training Set Only ==================== #
augmented_rows = []

for _, row in df_class_train.iterrows():
    variants = generate_multiple_paraphrases(row, n=3)
    for variant in variants:
        new_row = row.copy()
        new_row['Review'] = variant
        new_row['source'] = 'augmented'
        augmented_rows.append(new_row)

# ==================== Ensure DataFrame Structure is Preserved ==================== #
df_augmented = pd.DataFrame(augmented_rows)

# Ensure same column order and types
df_class_train_multiplied = pd.concat(
    [df_class_train, df_augmented],
    ignore_index=True
).sample(frac=1).reset_index(drop=True)

# ==================== Displaying Dataset Growth ==================== #
summary = pd.DataFrame({
    'Stage': ['Original Train Set', 'After Multiplication'],
    'Count': [len(df_class_train), len(df_class_train_multiplied)],
    'Increase': [0, len(df_class_train_multiplied) - len(df_class_train)]
})

summary.style.background_gradient(cmap="Blues")

Device set to use cuda:0


,Stage,Count,Increase
0,Original Train Set,222,0
1,After Multiplication,888,666


With reference to the output above, we are able to verify that the Data Multiplication have been done successfully and we are now able to move on to conduct Data Augmentation for the Regression Dataset.

---
### 4.3.3 Classification Duplication Check

As new data are generated, there will most likely be cases where Duplicated Data may be present. This will cause Model Training issues as the Model will re-learn data more than once which can cause the performance to be skewed. Therefore, we will be identifying for Duplicated Data within the Augmented Data in the Code Cell below.

In [ ]:
# ==================== Creating Duplicate Summary ==================== #
duplicate_count = df_class_train_multiplied.duplicated().sum()
summary_df = pd.DataFrame({'Duplicate Count': [duplicate_count]})

# ==================== Colour Coding Function ==================== #
def highlight_duplicates(val):
    color = 'red' if val > 0 else 'green'
    return f'background-color: {color}; color: white'

# ==================== Apply Styling ==================== #
styled_summary = summary_df.style.map(highlight_duplicates)

# ==================== Display Summary ==================== #
styled_summary

,Duplicate Count
0,39


With reference to the Code Cell above, we are able to see that there are indeed Duplicated Data. Therefore, we will have to address the Duplicated Data through removal but also to ensure Class Imbalance does not become an issue again.

---
### 4.3.4 Classification Duplicate Removal

As identified before, there are Duplicated Data within our Classification Data, therefore, we will need to identify the Classes with the Duplicated Data, drop them and subseqeuntly, add back new Data which will be translated from the other language (i.e. English to Malay) to fill in the Duplicated Data. The new Data that will be placed back in must be from the same class and from the opposite language. This operation will be done in the Code Cell below.

In [ ]:
# ==================== Dropping Duplicates ==================== #
df_class_train_multiplied = df_class_train_multiplied.drop_duplicates()

# ==================== Load translation models ==================== #
en_to_id_tokenizer = MarianTokenizer.from_pretrained("Helsinki-NLP/opus-mt-en-id")
en_to_id_model = MarianMTModel.from_pretrained("Helsinki-NLP/opus-mt-en-id")

id_to_en_tokenizer = MarianTokenizer.from_pretrained("Helsinki-NLP/opus-mt-id-en")
id_to_en_model = MarianMTModel.from_pretrained("Helsinki-NLP/opus-mt-id-en")

# ==================== Translation Functions ==================== #
def translate(text, tokenizer, model):
    try:
        batch = tokenizer.prepare_seq2seq_batch([text], return_tensors="pt", truncation=True)
        translated = model.generate(**batch)
        return tokenizer.decode(translated[0], skip_special_tokens=True)
    except:
        return None

def is_english(text):
    return all(ord(c) < 128 for c in text)

def back_translate(text):
    if is_english(text):
        malay = translate(text, en_to_id_tokenizer, en_to_id_model)
        return translate(malay, id_to_en_tokenizer, id_to_en_model) if malay else None
    else:
        english = translate(text, id_to_en_tokenizer, id_to_en_model)
        return translate(english, en_to_id_tokenizer, en_to_id_model) if english else None

# ==================== Balancing Logic with Language & Label ==================== #
class_counts = df_class_train_multiplied['Sentiment_Label'].value_counts()
max_count = class_counts.max()
augmented_rows = []

for label, count in class_counts.items():
    needed = max_count - count
    if needed <= 0:
        continue

    df_class = df_class_train_multiplied[df_class_train_multiplied['Sentiment_Label'] == label]
    gen_count = 0
    i = 0

    while gen_count < needed and i < len(df_class) * 10:
        row = df_class.iloc[i % len(df_class)]
        review = row['Review']
        new_review = back_translate(review)

        if new_review and new_review.strip() != "":
            new_row = row.copy()
            new_row['Review'] = new_review
            new_row['source'] = 'augmented'
            augmented_rows.append(new_row)
            gen_count += 1

        i += 1

# ==================== Combine with Original ==================== #
df_augmented = pd.DataFrame(augmented_rows)
df_class_train_multiplied = pd.concat([df_class_train_multiplied, df_augmented], ignore_index=True)

# ==================== Review Distribution Plot ==================== #
fig_lang = px.histogram(
    df_class_train_multiplied,
    x='Sentiment_Label',
    color='Sentiment_Label',
    title='Review Distribution',
    color_discrete_sequence=px.colors.qualitative.Bold
)

fig_lang.update_layout(
    xaxis_title='Sentiment_Label',
    yaxis_title='Number of Reviews',
    template='plotly_white',
    title_font=dict(size=20),
    showlegend=False
)

fig_lang.show()

/usr/local/lib/python3.11/dist-packages/transformers/models/marian/tokenization_marian.py:175: UserWarning:

Recommended: pip install sacremoses.

/usr/local/lib/python3.11/dist-packages/transformers/tokenization_utils_base.py:4106: FutureWarning:


`prepare_seq2seq_batch` is deprecated and will be removed in version 5 of HuggingFace Transformers. Use the regular
`__call__` method to prepare your inputs and targets.

Here is a short example:

model_inputs = tokenizer(src_texts, text_target=tgt_texts, ...)

If you either need to use different keyword arguments for the source and target texts, you should do two calls like
this:

model_inputs = tokenizer(src_texts, ...)
labels = tokenizer(text_target=tgt_texts, ...)
model_inputs["labels"] = labels["input_ids"]

See the documentation of your specific tokenizer for more details on the specific arguments to the tokenizer of choice.
For a more complete example, see the implementation of `prepare_seq2seq_batch`.




---
### 4.3.5 Regression Data Balancing Augmentation

For the Regression Dataset, we will be balancing the Skewed Data by increasing the Data Quantity for the higher scores. We will be achieving the balance through the usage of T5 Paraphraser and Deep-Translator. The T5 Paraphraser will be used as a paraphrasing model to augment underrepresented sentiment categories. It operates as a pretrained transformer, applying text-to-text transformation on input strings without altering labels or requiring external inference. The Deep-Translator will be used to translate the Reviews that are in Malay to English and subsequently, back to Malay. The Deep Translator is used as the T5 Paraphraser only supports English. The Balancing Data Augmentation will be conducted in the Code Cell below.

In [ ]:
# ==================== Load T5 Paraphrasing Model ==================== #
paraphraser = pipeline("text2text-generation", model="Vamsi/T5_Paraphrase_Paws", max_length=100)

# ==================== Language-aware Paraphrasing Function ==================== #
def stacked_paraphrase(text, lang):
    try:
        if lang.lower() == 'english':
            prompt = f"paraphrase: {text} </s>"
            return paraphraser(prompt, num_return_sequences=1, do_sample=True)[0]['generated_text']
        elif lang.lower() == 'malay':
            to_en = GoogleTranslator(source='auto', target='en').translate(text)
            prompt = f"paraphrase: {to_en} </s>"
            paraphrased = paraphraser(prompt, num_return_sequences=1, do_sample=True)[0]['generated_text']
            return GoogleTranslator(source='auto', target='ms').translate(paraphrased)
        else:
            return text
    except:
        return text

# ==================== Score Band Splitting (on TRAIN ONLY) ==================== #
df_high = df_reg_train[df_reg_train['Score'] < 0.4]
df_mid  = df_reg_train[(df_reg_train['Score'] >= 0.4) & (df_reg_train['Score'] < 0.7)]
df_low  = df_reg_train[df_reg_train['Score'] >= 0.7]

max_size = max(len(df_low), len(df_mid), len(df_high))
augmented_rows = []

for df_bin in [df_mid, df_high]:
    needed = max_size - len(df_bin)
    samples = df_bin.sample(n=needed, replace=True).to_dict(orient='records')
    for row in samples:
        row['Review'] = stacked_paraphrase(row['Review'], row['Language'])
        row['source'] = 'augmented'
        augmented_rows.append(row)

# ==================== Combine and Shuffle Training Set ==================== #
df_reg_train_balanced = pd.concat([df_reg_train, pd.DataFrame(augmented_rows)], ignore_index=True).sample(frac=1).reset_index(drop=True)

df_reg_train_balanced['Score Band'] = df_reg_train_balanced['Score'].apply(
    lambda s: 'Low' if s < 0.4 else 'Mid' if s < 0.7 else 'High'
)

df_reg_train_balanced['Score Band'].value_counts()

Device set to use cuda:0


,count
Score Band,
Low,167
High,167
Mid,167


With reference to the output above, we are able to see that the Data have been successfully balanced and we can now proceed to the next Data Augmentation on the Regression Dataset to multiply it in quantity to provide larger amounts of Data for Model Training.

---
### 4.3.6 Regression Multiplication Data Augmentation

Now that the Data for each of the Score Bins have been balanced, we will now increase the Data Volume for all the Score Bins so as to increase the Learning Capabilities of the Model. This will be achieved through using the T5 Paraphraser and Deep-Translator but this time for a different purpose which is to Multiply instead of Balancing. We will be multiplying the Data by 5 Times

In [ ]:
# ==================== Load T5 Paraphrasing Model ==================== #
paraphraser = pipeline("text2text-generation", model="Vamsi/T5_Paraphrase_Paws", max_length=100)

# ==================== Language-Aware Paraphrasing Function ==================== #
def generate_multiple_paraphrases(row, n=3):
    text = row['Review']
    lang = row['Language']
    try:
        if lang.lower() == 'english':
            results = paraphraser(f"paraphrase: {text} </s>", num_return_sequences=n, do_sample=True, top_k=50, top_p=0.95)
            return [res['generated_text'] for res in results]
        elif lang.lower() == 'malay':
            to_en = GoogleTranslator(source='auto', target='en').translate(text)
            results = paraphraser(f"paraphrase: {to_en} </s>", num_return_sequences=n, do_sample=True, top_k=50, top_p=0.95)
            return [GoogleTranslator(source='auto', target='ms').translate(res['generated_text']) for res in results]
        else:
            return [text] * n
    except:
        return [text] * n

# ==================== Multiply Training Set Only ==================== #
augmented_rows = []

for _, row in df_reg_train_balanced.iterrows():
    variants = generate_multiple_paraphrases(row, n=3)
    for variant in variants:
        new_row = row.copy()
        new_row['Review'] = variant
        new_row['source'] = 'augmented'
        augmented_rows.append(new_row)

# ==================== Combine and Shuffle ==================== #
df_augmented = pd.DataFrame(augmented_rows)

# Ensure consistent column structure with original training DataFrame
df_reg_train_multiplied = pd.concat(
    [df_reg_train_balanced, df_augmented],
    ignore_index=True
).sample(frac=1).reset_index(drop=True)

# ==================== Displaying Dataset Growth ==================== #
summary = pd.DataFrame({
    'Stage': ['After Balancing', 'After Multiplication'],
    'Sample Count': [len(df_reg_train_balanced), len(df_reg_train_multiplied)],
    'Increase': [0, len(df_reg_train_multiplied) - len(df_reg_train_balanced)]
})

# ==================== Displaying DataFrame ==================== #
summary.style.background_gradient(cmap="Blues")

Device set to use cuda:0


,Stage,Sample Count,Increase
0,After Balancing,501,0
1,After Multiplication,2004,1503


With reference to the output above, we are able to verify that the Data Multiplication have been done successfully and we are now able to move on to conduct the other Data Pre-Processing steps.

---
### 4.3.7 Regression Duplication Check

As new data are generated, there will most likely be cases where Duplicated Data may be present. This will cause Model Training issues as the Model will re-learn data more than once which can cause the performance to be skewed. Therefore, we will be identifying for Duplicated Data within the Augmented Data in the Code Cell below.

In [ ]:
# ==================== Creating Duplicate Summary ==================== #
duplicate_count = df_reg_train_multiplied.duplicated().sum()
summary_df = pd.DataFrame({'Duplicate Count': [duplicate_count]})

# ==================== Colour Coding Function ==================== #
def highlight_duplicates(val):
    color = 'red' if val > 0 else 'green'
    return f'background-color: {color}; color: white'

# ==================== Apply Styling ==================== #
styled_summary = summary_df.style.map(highlight_duplicates)

# ==================== Display Summary ==================== #
styled_summary

,Duplicate Count
0,459


With reference to the Code Cell above, we are able to see that there are indeed Duplicated Data. Therefore, we will have to address the Duplicated Data through removal but also to ensure Class Imbalance does not become an issue again.

---
### 4.3.8 Regression Duplicate Removal

As identified before, there are Duplicated Data within our Classification Data, therefore, we will need to identify the Classes with the Duplicated Data, drop them and subseqeuntly, add back new Data which will be translated from the other language (i.e. English to Malay) to fill in the Duplicated Data. The new Data that will be placed back in must be from the same class and from the opposite language. This operation will be done in the Code Cell below.

In [ ]:
# ==================== Drop Duplicates ==================== #
df_reg_train_multiplied = df_reg_train_multiplied.drop_duplicates()

# ==================== Load Translation Models ==================== #
en_to_id_tokenizer = MarianTokenizer.from_pretrained("Helsinki-NLP/opus-mt-en-id")
en_to_id_model = MarianMTModel.from_pretrained("Helsinki-NLP/opus-mt-en-id")

id_to_en_tokenizer = MarianTokenizer.from_pretrained("Helsinki-NLP/opus-mt-id-en")
id_to_en_model = MarianMTModel.from_pretrained("Helsinki-NLP/opus-mt-id-en")

# ==================== Translation Utilities ==================== #
def translate(text, tokenizer, model):
    try:
        batch = tokenizer.prepare_seq2seq_batch([text], return_tensors="pt", truncation=True)
        translated = model.generate(**batch)
        return tokenizer.decode(translated[0], skip_special_tokens=True)
    except:
        return None

def is_english(text):
    return all(ord(c) < 128 for c in text)

def back_translate(text):
    if is_english(text):
        malay = translate(text, en_to_id_tokenizer, en_to_id_model)
        return translate(malay, id_to_en_tokenizer, id_to_en_model) if malay else None
    else:
        english = translate(text, id_to_en_tokenizer, id_to_en_model)
        return translate(english, en_to_id_tokenizer, en_to_id_model) if english else None

# ==================== Score Banding ==================== #
def get_score_band(score):
    if score < 0.4:
        return 'High'
    elif score < 0.7:
        return 'Mid'
    else:
        return 'Low'

df_reg_train_multiplied['Score Band'] = df_reg_train_multiplied['Score'].apply(get_score_band)

# ==================== Balancing by Score Band ==================== #
band_counts = df_reg_train_multiplied['Score Band'].value_counts()
max_count = band_counts.max()

augmented_rows = []

for band in ['Low', 'Mid', 'High']:
    current_count = band_counts.get(band, 0)
    needed = max_count - current_count
    if needed <= 0:
        continue

    df_band = df_reg_train_multiplied[df_reg_train_multiplied['Score Band'] == band]

    gen_count = 0
    i = 0
    while gen_count < needed and i < len(df_band) * 10:
        row = df_band.iloc[i % len(df_band)]
        review = row['Review']
        new_review = back_translate(review)

        if new_review and new_review.strip() != "":
          new_row = row.copy()
          new_row['Review'] = new_review
          new_row['source'] = 'augmented'
          augmented_rows.append(new_row)
          gen_count += 1

        i += 1

# ==================== Combine Final Dataset ==================== #
df_augmented = pd.DataFrame(augmented_rows)
df_reg_train_multiplied = pd.concat([df_reg_train_multiplied, df_augmented], ignore_index=True)

# ==================== Refresh Score Band (Optional Redundancy) ==================== #
df_reg_train_multiplied['Score Band'] = df_reg_train_multiplied['Score'].apply(get_score_band)

# ==================== Visualise Distribution ==================== #
fig_score = px.histogram(
    df_reg_train_multiplied,
    x='Score',
    nbins=30,
    marginal='box',
    color_discrete_sequence=['#1f77b4'],
    title='Distribution of Sentiment Scores (Regression Target)'
)

fig_score.update_layout(
    xaxis_title='Sentiment Score (0 to 1)',
    yaxis_title='Number of Reviews',
    template='plotly_white',
    title_font=dict(size=20),
)
fig_score.show()

/usr/local/lib/python3.11/dist-packages/transformers/models/marian/tokenization_marian.py:175: UserWarning:

Recommended: pip install sacremoses.

/usr/local/lib/python3.11/dist-packages/transformers/tokenization_utils_base.py:4106: FutureWarning:


`prepare_seq2seq_batch` is deprecated and will be removed in version 5 of HuggingFace Transformers. Use the regular
`__call__` method to prepare your inputs and targets.

Here is a short example:

model_inputs = tokenizer(src_texts, text_target=tgt_texts, ...)

If you either need to use different keyword arguments for the source and target texts, you should do two calls like
this:

model_inputs = tokenizer(src_texts, ...)
labels = tokenizer(text_target=tgt_texts, ...)
model_inputs["labels"] = labels["input_ids"]

See the documentation of your specific tokenizer for more details on the specific arguments to the tokenizer of choice.
For a more complete example, see the implementation of `prepare_seq2seq_batch`.




With reference to the output above, we are able to see that the Data have been successfully balanced and we can now proceed to the next Data Augmentation on the Classification Dataset to multiply it in quantity to provide larger amounts of Data for Model Training.

---
## 4.4 Lowercasing Reviews

In this sub-section, we will be making all the Reviews lowercased. This will help to reduce variability and complexity in the text data, simplifying later processing steps and potentially improving the performance of the RNN models. This operation will be conducted in the Code Cell below.

In [ ]:
# ==================== Lowercasing Review Text ==================== #
df_class_train_multiplied['Review'] = df_class_train_multiplied['Review'].str.lower()
df_reg_train_multiplied['Review'] = df_reg_train_multiplied['Review'].str.lower()

# ==================== Displaying DataFrames ==================== #
print("Classification DataFrame:")
display(df_class_train_multiplied.head(5).style.background_gradient(cmap="Blues"))
print("\n"*5)
print("Regression DataFrame:")
display(df_reg_train_multiplied.head(5).style.background_gradient(cmap="Blues"))

Classification DataFrame:


,Review,Language,word_count,Sentiment_Binary,Sentiment_Label,Score,source
0,the visual effects of this film are outstanding ; excited about the technical brilliance,English,13,0,Positive,0.900000,augmented
1,"while the film attempts to tie up loose ends, certain character resolutions may leave fans divided. however, avengers: endgame excels in the epic battles and jaw-dropping visuals - a must for those who invested in the marvel cinematic universe .",English,39,0,Positive,0.750000,augmented
2,the visual effects and animation in this film are amazing. impressed!,English,11,0,Positive,0.925000,original
3,kekurangan perkembangan watak menjadikan filem ini lemah dan tidak memuaskan.,Malay,9,1,Negative,0.250000,augmented
4,"the open-ended nature of avengers: endgame's plot allows for interpretation and speculation. while some may appreciate the ambiguity, others may find it leaves too much unanswered. the film leaves room for potential future developments in the marvel cinematic universe.",English,39,1,Negative,0.300000,original








Regression DataFrame:


,Review,Language,word_count,Sentiment_Binary,Score,source,Score Band
0,my problem is that the plot of this film is too complex and a bit difficult to follow.,English,16,1,0.350000,augmented,High
1,"for me, the plot of this film is too complex, a bit difficult to follow .",English,16,1,0.350000,augmented,High
2,"dramatic and impactful action scenes , too many special effects .",English,9,1,0.400000,augmented,Mid
3,too many over-the-top action scenes. too many special effects,English,9,1,0.400000,augmented,Mid
4,"adegan tindakan dramatik dan tidak dapat dilupakan, terlalu banyak kesan biasa.",Malay,10,1,0.400000,augmented,Mid


With reference to the output of the DataFrame above, we are able to see that the Review have all been successfully set to lowercase.

---
## 4.5 Removing Punctuations

In this sub-section, we will be removing the Punctuations within each of the review. This is done so as to remove unneccesary noise and text complexity within the Review Data Column. This will also help to simplify the Spliting and Processing of the Text.

In [ ]:
# ==================== Removing Punctuations Function ==================== #
def remove_punctuation(text):
    return re.sub(r'[^\w\s]', '', text)

# ==================== Classification: Apply to All Splits ==================== #
df_class_train_multiplied['Review'] = df_class_train_multiplied['Review'].apply(remove_punctuation)
df_class_train['Review'] = df_class_train['Review'].apply(remove_punctuation)
df_class_val['Review'] = df_class_val['Review'].apply(remove_punctuation)
df_class_test['Review'] = df_class_test['Review'].apply(remove_punctuation)

# ==================== Regression: Apply to All Splits ==================== #
df_reg_train_multiplied['Review'] = df_reg_train_multiplied['Review'].apply(remove_punctuation)
df_reg_train['Review'] = df_reg_train['Review'].apply(remove_punctuation)
df_reg_val['Review'] = df_reg_val['Review'].apply(remove_punctuation)
df_reg_test['Review'] = df_reg_test['Review'].apply(remove_punctuation)

With reference to the Code Cell above, we are able to verify that the Punctuations have been successfully removed for both of the Dataset.

---
## 4.6 Stopwords Removal

In this sub-section, we will be removing all the Stopwords which are present within each of the Review. This will remove all word which carry insignificant meaning and prevent unneccesary noise within our data. This operation will be conducted in the Code Cell below.

In [ ]:
# ==================== Setting Stopwords ==================== #
stop_words_en = set(stopwords.words('english'))

# ==================== Malay Stopwards ==================== #
stop_words_ms = set([
    "yang", "dan", "di", "ke", "dari", "itu", "untuk", "ini", "pada", "adalah", "dengan", "juga", "akan",
    "saya", "kami", "mereka", "anda", "tidak", "ya", "boleh", "harus", "kerana", "lagi", "kalau"
])

# ==================== Removing Stopwords Function ==================== #
def remove_stopwords(text, lang):
    tokens = word_tokenize(text.lower())
    if lang.lower() == 'english':
        filtered = [word for word in tokens if word not in stop_words_en]
    elif lang.lower() == 'malay':
        filtered = [word for word in tokens if word not in stop_words_ms]
    else:
        filtered = tokens
    return ' '.join(filtered)

# ==================== Applying Stopwords Function For Classification ==================== #
df_class_train_multiplied['Review'] = df_class_train_multiplied.apply(
    lambda row: remove_stopwords(row['Review'], row['Language']), axis=1
)

df_class_train['Review'] = df_class_train.apply(
    lambda row: remove_stopwords(row['Review'], row['Language']), axis=1
)

df_class_val['Review'] = df_class_val.apply(
    lambda row: remove_stopwords(row['Review'], row['Language']), axis=1
)

df_class_test['Review'] = df_class_test.apply(
    lambda row: remove_stopwords(row['Review'], row['Language']), axis=1
)

# ==================== Applying Stopwords Function For Regression ==================== #
df_reg_train_multiplied['Review'] = df_reg_train_multiplied.apply(
    lambda row: remove_stopwords(row['Review'], row['Language']), axis=1
)

df_reg_train['Review'] = df_reg_train.apply(
    lambda row: remove_stopwords(row['Review'], row['Language']), axis=1
)

df_reg_val['Review'] = df_reg_val.apply(
    lambda row: remove_stopwords(row['Review'], row['Language']), axis=1
)

df_reg_test['Review'] = df_reg_test.apply(
    lambda row: remove_stopwords(row['Review'], row['Language']), axis=1
)

With reference to the Code Cell above, we are able to verify that the Removal of Stop Words have been done successfully and we are able to move on to the next Data Pre-Processing step.

---
## 4.7 Data Lemmatisation

In this sub-section, we will be performing Data Lemmatisation which will change all the remaining words into their base or dictionary form. This will make our data more consistent and is also known to make the RNN Model perform much better in terms of accuracy. We will only perfrom Lemmatisation on our English Data and will not perform it on the Malay data as there is a lack of accessible and reliable lemmatisation tools for Malay in the development environment. Therefore, we will omit this step for Malay reviews.

In [ ]:
# ==================== Lemmatisation Function ==================== #
lemmatizer = WordNetLemmatizer()

def lemmatise_text(text, lang):
    tokens = word_tokenize(text)
    if lang.lower() == 'english':
        lemmatised = [lemmatizer.lemmatize(word) for word in tokens]
        return ' '.join(lemmatised)
    else:
        return text

# ==================== Applying Lemmatisation Function For Classification ==================== #
df_class_train_multiplied['Review'] = df_class_train_multiplied.apply(
    lambda row: lemmatise_text(row['Review'], row['Language']), axis=1
)

df_class_train['Review'] = df_class_train.apply(
    lambda row: lemmatise_text(row['Review'], row['Language']), axis=1
)

df_class_val['Review'] = df_class_val.apply(
    lambda row: lemmatise_text(row['Review'], row['Language']), axis=1
)

df_class_test['Review'] = df_class_test.apply(
    lambda row: lemmatise_text(row['Review'], row['Language']), axis=1
)

# ==================== Applying Lemmatisation Function For Regression ==================== #
df_reg_train_multiplied['Review'] = df_reg_train_multiplied.apply(
    lambda row: lemmatise_text(row['Review'], row['Language']), axis=1
)

df_reg_train['Review'] = df_reg_train.apply(
    lambda row: lemmatise_text(row['Review'], row['Language']), axis=1
)

df_reg_val['Review'] = df_reg_val.apply(
    lambda row: lemmatise_text(row['Review'], row['Language']), axis=1
)

df_reg_test['Review'] = df_reg_test.apply(
    lambda row: lemmatise_text(row['Review'], row['Language']), axis=1
)

With reference to the Code Cell above, we are able to verify that the Lemmatisation have been done successfully and we are able to move on to the next Data Pre-Processing step.


---
## 4.8 Fitting Shared Tokeniser

In this sub-section, we will be using a Shared Tokeniser on our Data so as to make the review textual data understandable and usable for the RNN Models. We will be using a Shared Tokeniser so as to ensure consistency throughout both of our Data. The Tokenisation will be conducted in the Code Cell below.

In [ ]:
# ==================== Combine All Review Texts ==================== #
all_reviews = pd.concat([
    df_class_train_multiplied['Review'],
    df_class_train['Review'],
    df_class_val['Review'],
    df_class_test['Review'],
    df_reg_train_multiplied['Review'],
    df_reg_train['Review'],
    df_reg_val['Review'],
    df_reg_test['Review']
], ignore_index=True)

# ==================== Initialise and Fit Tokeniser ==================== #
vocab_size = 10000
tokenizer = Tokenizer(num_words=vocab_size, oov_token="<OOV>")
tokenizer.fit_on_texts(all_reviews)

With reference to the code cell above, we are able to verify that the Tokenisation Process have been completed.

---
## 4.9 Converting Text To Sequences

In this sub-section, we will be converting the Textual Data of the Movie Reviews to Sequences which will help to maintain the syntactical meaning of a sentence. This will allow the RNN Model understand the Input Data as it will be able to analyse and manipulate the data. This operation will be conducted in the Code Cell below.

In [ ]:
# ==================== Classification Splits ==================== #
X_class_train = tokenizer.texts_to_sequences(df_class_train_multiplied['Review'])
X_class_train_no_aug = tokenizer.texts_to_sequences(df_class_train['Review'])
X_class_val = tokenizer.texts_to_sequences(df_class_val['Review'])
X_class_test = tokenizer.texts_to_sequences(df_class_test['Review'])

# ==================== Regression Splits ==================== #
X_reg_train = tokenizer.texts_to_sequences(df_reg_train_multiplied['Review'])
X_reg_train_no_aug = tokenizer.texts_to_sequences(df_reg_train['Review'])
X_reg_val = tokenizer.texts_to_sequences(df_reg_val['Review'])
X_reg_test = tokenizer.texts_to_sequences(df_reg_test['Review'])

With reference to the Code Cell above, we are able to verify that the conversion have been successfully executed.

---
## 4.10 Padding Sequences

In this sub-section, we will be adding Padding to our Sequences. This is to ensure that input sequences have the same length by padding them with a specific value or truncating them if they are too long. As previously identified in our Exploratory Data Analysis, the best max length to utilise is 100. Therefore, the max length of 100 will be utilised in the Code Cell below.

In [ ]:
# ==================== Padding Configuration ==================== #
maxlen = 100

# ==================== Classification Padded ==================== #
X_train_class_pad = pad_sequences(X_class_train, maxlen=maxlen, padding='post', truncating='post')
X_train_class_pad_no_aug = pad_sequences(X_class_train_no_aug, maxlen=maxlen, padding='post', truncating='post')
X_val_class_pad = pad_sequences(X_class_val,   maxlen=maxlen, padding='post', truncating='post')
X_test_class_pad = pad_sequences(X_class_test,  maxlen=maxlen, padding='post', truncating='post')

# ==================== Regression Padded ==================== #
X_train_reg_pad = pad_sequences(X_reg_train, maxlen=maxlen, padding='post', truncating='post')
X_train_reg_pad_no_aug = pad_sequences(X_reg_train_no_aug, maxlen=maxlen, padding='post', truncating='post')
X_val_reg_pad = pad_sequences(X_reg_val,   maxlen=maxlen, padding='post', truncating='post')
X_test_reg_pad = pad_sequences(X_reg_test,  maxlen=maxlen, padding='post', truncating='post')

With reference to the Code Cell above, we are able to verify that the Padding Sequence have been successfully executed and we can proceed to the next Pre-Processing step.

---
## 4.11 Preparing Target Variables

In this sub-section, we will be creating a separate variable for the Target that will we plan to get the RNN to output. There will be two separate targets as we will be doing both Classification and Regression and comparing them afterwards. The preparation will be done in the Code Cell below.

In [ ]:
# ==================== Classification Targets ==================== #
y_train_class = df_class_train_multiplied['Sentiment_Binary'].values
y_val_class = df_class_val['Sentiment_Binary'].values
y_test_class = df_class_test['Sentiment_Binary'].values

# ==================== Regression Targets ==================== #
y_train_reg = df_reg_train_multiplied['Score'].values
y_val_reg = df_reg_val['Score'].values
y_test_reg = df_reg_test['Score'].values

With reference to the Code Cell above, we are able to verify that the preparation of Target Variables have been successful.